# Where does a volcano get its magma?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F11_where_magma.ipynb).

A volcano is a hole in the ground with hot rock coming out of it, and the three commonest kinds of
basaltic volcano on Earth are fed in three completely different ways. Under a mid-ocean ridge the
mantle rises, the pressure on it drops, and it melts on its own. Under an island arc the mantle is
too cold to melt at all until water squeezed out of a sinking plate lowers its melting point. Under
an ocean island neither of those applies, and something hotter appears to be arriving from deeper
down. Three plumbing systems, three magmas — and once the lava has cooled into black basalt on a
beach, the three look much alike.

Today you get 756 basalts whose setting somebody has already established, with up to 51 chemical
measurements on each, and one question: is the setting written in the chemistry? On the way you
will find out what a model does when half of those measurements were never made.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Name the three tectonic settings basalt comes from and the melting mechanism
behind each, say which chemical elements separate them and why the petrology predicts those
elements, and state how much of a classifier's skill is chemistry and how much is bookkeeping.

**The skills.** Two new classifiers, both three lines behind the same `fit` / `score` interface you
met with logistic regression: `DecisionTreeClassifier`, `RandomForestClassifier`, and
`SVC` alongside them. `SimpleImputer` to fill holes instead of deleting rows,
`StandardScaler` to put columns on a common footing, and `feature_importances_` to ask a trained
model which columns it actually used.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

**The four questions, in order:**

1. Can two elements tell three tectonic settings apart?
2. What do you do when no column in the file is complete?
3. Do the badly measured columns help, or hurt?
4. Is the forest reading the chemistry, or reading who measured the rock?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

# the Vermeesch (2006) compilation, which ships with the course, so there is nothing to fetch
basalts = pd.read_csv(CACHE + "/week11_vermeesch_basalts.csv")

feature_columns = []
for name in basalts.columns:
    if name != "affinity":
        feature_columns.append(name)

print(basalts.shape, "-", len(feature_columns), "chemistry columns and one label")

## Can two elements tell three tectonic settings apart?

Basalt is the most common lava on the planet, and almost all of it erupts in one of three places.

**Mid-ocean ridge basalt (MORB)** comes from a spreading centre. The mantle beneath rises, the
pressure on it falls, and it melts without anything being added to it. That mantle has already had
melt extracted from it once before, so what comes out is poor in the elements that leave easily —
potassium, rubidium, barium.

**Island arc basalt (IAB)** comes from above a sinking plate. The mantle in the wedge there is dry,
and dry mantle at that temperature does not melt. What makes it melt is water: the sinking plate
heats up, releases water, and water lowers the melting point of rock. The water carries with it
whatever dissolves in water — potassium, rubidium, barium, strontium, lead — and leaves behind
whatever does not, notably niobium, tantalum and titanium. Arc basalts therefore carry a
distinctive signature: enriched in the first group, unusually poor in the second.

**Ocean island basalt (OIB)** is Hawaii and Iceland. Neither mechanism applies; the mantle arriving
underneath is hotter and has been through less, and the melt fractions are small, so the lava is
rich in titanium, niobium and the light rare earth elements.

The table you have loaded is the compilation from Vermeesch, P. (2006), *Tectonic discrimination
of basalts with classification trees*, Geochimica et Cosmochimica Acta 70, 1839–1848
(doi:10.1016/j.gca.2005.12.016). Somebody established the setting of each sample from where it was
collected; the `affinity` column is that answer.

In [ ]:
print(basalts[["affinity", "SiO2_wt_percent", "TiO2_wt_percent", "Sr_ppm"]].head())
print(basalts["affinity"].value_counts())

Three classes, and near enough the same number of each — which is a piece of luck, because it means
an accuracy here means what it looks like it means. The class-imbalance trap from the
classification week does not bite.

Before any model, the dumbest rule you can write. *Write the dumbest rule you can, first. Any model
that cannot beat it is decoration.* Here that rule is: ignore the chemistry entirely and call every
sample whichever class is commonest.

### ✏️ Your turn 1

How often would that rule be right? Take `basalts["affinity"].value_counts()`, ask it for its
largest value with `.max()`, and divide by the number of rows. Print the result rounded to three
decimal places.

**Use these names**, because the self-check looks for them: `baseline`.

In [ ]:
# ← your answer here


assert baseline < 1, "baseline is a fraction of 1, not a count of rows"
print("✓ the baseline — guessing the commonest of the three settings is right",
      round(baseline * 100, 1), "percent of the time")

Geochemists have been separating these three settings by hand since the 1970s, and they did it with
two elements at a time on a piece of graph paper. Pearce, J.A. and Cann, J.R. (1973), *Tectonic
setting of basic volcanic rocks determined using trace element analyses*, Earth and Planetary
Science Letters 19, 290–300, used titanium, zirconium and yttrium in one diagram and titanium,
zirconium and strontium in another; Shervais, J.W. (1982), *Ti-V plots and the petrogenesis of
modern and ophiolitic lavas*, Earth and Planetary Science Letters 59, 101–118, used titanium
against vanadium. Start where they did.

In [ ]:
pairs = basalts[["TiO2_wt_percent", "V_ppm", "affinity"]].dropna()

for setting in ["MORB", "OIB", "IAB"]:
    rows = pairs[pairs["affinity"] == setting]
    plt.scatter(rows["TiO2_wt_percent"], rows["V_ppm"], s=12, label=setting)

plt.xlabel("TiO2 (weight percent)")
plt.ylabel("V (ppm)")
plt.title("514 basalts with both TiO2 and V measured")
plt.legend()
plt.show()

The three settings do sit in different places, and they overlap badly. Almost all of the separation
runs left to right: the three clouds sit at three different titanium levels, while their vanadium
ranges lie on top of one another — the three median vanadium values are within
18 ppm of each other, so vanadium on its own would tell you almost nothing
here. That does not make it decoration. It is the second direction a boundary can cut in, and where
the clouds meet it is the only one left. You could draw a boundary on that picture with a ruler, and
you would get a lot of samples wrong.

A **decision tree** draws the boundary for you. A flowchart of yes/no questions, learned from the
data instead of written by hand. *Is TiO2 above some value?* If it is, *is V below another?* — and
so on, with both the element and the threshold at every step chosen to separate the three classes
as cleanly as they can be separated, until every branch ends in a verdict. On a two-element plot
that comes out as a staircase of horizontal and vertical cuts.

The machinery is the machinery from the classification week: split the samples into a training set
and a held-out test set, fit on the first, score on the second. One thing is new. Every column in
this file has holes in it, and no classifier in scikit-learn will accept a hole, so for now we do
the obvious thing and throw away every row that has one — `.dropna()`, from the tables week. Note
what that costs: 514 of the 756 samples survive.

In [ ]:
X = pairs[["TiO2_wt_percent", "V_ppm"]]
y = pairs["affinity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0,
                                                    stratify=y)

tree = DecisionTreeClassifier(random_state=0)
tree.fit(X_train, y_train)
ti_v_score = tree.score(X_test, y_test)

print("rows kept:", len(pairs), "of", len(basalts))
print("test accuracy:", round(ti_v_score, 3))

### ✏️ Your turn 2

Two of Pearce and Cann's three elements now, instead of Shervais's pair. Do exactly what the cell
above did, but for `Zr_ppm` and `Y_ppm`: drop the rows where either is blank, split with
`test_size=0.3`,
`random_state=0` and `stratify=`, fit a `DecisionTreeClassifier(random_state=0)`, and print how
many rows survived and the test accuracy.

**Use these names**, because the self-check looks for them: `zr_y` for the surviving rows, and
`zr_y_score` for the accuracy.

In [ ]:
# ← your answer here


assert "Zr_ppm" in zr_y.columns, "zr_y should be built from Zr and Y, not from TiO2 and V"
assert 0.5 < zr_y_score < 1, \
    "1.000 would mean you scored the tree on the rows it was fitted on"
print("✓ zirconium and yttrium —", len(zr_y), "rows survived and the tree scored",
      round(zr_y_score, 3))

Two numbers, both well clear of the 0.343 baseline, so there really is tectonic
information in two elements. But look at what you had to do to get them. The titanium-vanadium pair
was scored on 514 samples and the zirconium-yttrium pair on 576, and those are
not the same 576 rocks. Two accuracies measured on two different sets of samples cannot
be compared, and we are about to want to compare a dozen of them.

## What do you do when no column in the file is complete?

So how holey is this file? `.isna()` marks every hole with `True`, `.sum()` adds up the `True`s
column by column, and dividing by the number of rows turns each count into a fraction. The result
is one number per column, labelled with the column's name, so `missing["Sr_ppm"]` reads one of them
out.

In [ ]:
missing = basalts[feature_columns].isna().sum() / len(basalts)

print("best measured:")
print(missing.sort_values().head(3))
print("worst measured:")
print(missing.sort_values(ascending=False).head(3))

In [ ]:
plt.bar(range(len(missing)), missing.sort_values() * 100)
plt.xlabel("the 51 chemistry columns, best measured first")
plt.ylabel("percent of samples with no value")
plt.title("Missing measurements, 51 columns of 756 basalts")
plt.show()

That is not a chart with a few gaps in it. No bar touches zero, and by the middle of the chart half
the measurements have gone. The reason is that this is a *compilation*: hundreds of published
analyses of different rocks by different laboratories, and each of those studies measured whatever
its own question needed. The bulk chemistry of a rock comes off a single prepared bead in one run
and is cheap; a rare earth element needs a separate and more expensive technique; a lead isotope
ratio needs chemical separation and a mass spectrometer, sample by sample. Nobody was withholding
anything. The measurement was simply never made.

### ✏️ Your turn 3

Three counts, from `missing` and from the table itself.

1. How many of the columns have at least one hole in them — `missing[name] > 0`?
2. How many are more than half empty — `missing[name] > 0.5`?
3. How many rows survive `basalts.dropna()`, which throws away every row with a hole anywhere in
   it?

One loop over `feature_columns` with two counters will do the first two. Print all three.

**Use these names**, because the self-check looks for them: `n_with_holes`, `n_half_empty`,
`rows_left`.

In [ ]:
# ← your answer here


assert n_half_empty > 0, "n_half_empty came out 0 - 'missing' is a fraction, so half empty is 0.5"
assert n_with_holes >= n_half_empty, \
    "a half-empty column is a column with holes - the first count cannot be the smaller"
print("✓ the holes —", n_with_holes, "of the", len(feature_columns),
      "columns have holes in them,", n_half_empty, "are more than half empty, and dropna() leaves",
      rows_left, "of", len(basalts), "rows")

Not one column of the 51 is complete. The best measured is
`Sr_ppm` and even that is missing 6.5 percent of the time; the
worst is `Sn_ppm` at 97.1 percent. Insist on rows with nothing
missing anywhere and you are left with a single sample.

So `.dropna()` is finished as a strategy, and the alternative has a name. **Imputation.** A blank
is not a zero. Fill it with something defensible and say what you filled it with. Ours will be
`SimpleImputer(strategy="median")`, which puts the middle value of a column into that column's
holes — defensible because it does not invent an extreme, and honest because we are about to say
out loud that we did it.

Two things go with it. The filler learns its medians from the **training** set only and then
applies them to the test set, because a median computed from data you are about to be tested on is
the leakage trap from the model-selection week. And a `StandardScaler` puts every column on the
same footing afterwards, which changes nothing for a tree — a tree only asks whether a value is
above a threshold — but matters enormously for the third model further down.

That is five steps, and we are about to run them on a dozen different sets of columns, so write
them once, as a function, and hand it the model and the columns.

In [ ]:
def accuracy(model, features, seed=0):
    """Fit one model on one set of columns and score it on the held-out third of the samples."""
    labels = basalts["affinity"]
    # Split first, before anything at all is measured from the data. `stratify` keeps the mix
    # of affinities the same in both halves, so a rare one cannot land entirely in the test set.
    X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.3,
                                                        random_state=seed, stratify=labels)
    # Fill the holes, then put the columns on a common footing. Both of these learn their
    # numbers from the training half alone — `fit_transform` on it, plain `transform` on the
    # test half — because a median taken from rows you are about to be tested on is leakage.
    filler = SimpleImputer(strategy="median")
    X_train = filler.fit_transform(X_train)
    X_test = filler.transform(X_test)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    # Learn from the training half; report the score on the test half, which it has never seen.
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

In [ ]:
filled_score = accuracy(DecisionTreeClassifier(random_state=0),
                        basalts[["TiO2_wt_percent", "V_ppm"]])

print("TiO2 and V, dropping incomplete rows:", round(ti_v_score, 3), "on", len(pairs), "samples")
print("TiO2 and V, filling the holes instead:", round(filled_score, 3), "on", len(basalts),
      "samples")

The same two elements and the same tree score 0.727 rather than
0.832 once every sample is included — and those two are exactly the kind of pair you
were just told not to compare, because the second was tested on rocks the first never saw. Which is
the point. Nothing about the rocks changed; what changed is which rocks were allowed into the exam.
The 242 extra samples are exactly the ones with a hole in titanium or vanadium, and
they have arrived with a median in place of a measurement.

So where did that 0.105 go? Part of it is those samples, and you can see how much by
scoring the filled tree separately on the two kinds of rock in its own test set: the
147 test rocks that were really measured score 0.755, the
80 that arrived carrying a median score 0.675. The filled-in
rocks are genuinely harder, and that is the honest half of the story — `.dropna()` was not solving
that difficulty, it was hiding it, by quietly grading itself on the easy rocks only. But
80 rocks out of 227 cannot move an average by 0.105.
They account for 0.028 of it. The other 0.077 has nothing to do
with holes at all — it is the single split. Check that rather than believe it: fit both trees again
on ten different splits.

In [ ]:
for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(pairs[["TiO2_wt_percent", "V_ppm"]],
                                                        pairs["affinity"], test_size=0.3,
                                                        random_state=seed,
                                                        stratify=pairs["affinity"])
    dropped = DecisionTreeClassifier(random_state=0)
    dropped.fit(X_train, y_train)
    dropna_score = dropped.score(X_test, y_test)
    filled_again = accuracy(DecisionTreeClassifier(random_state=0),
                            basalts[["TiO2_wt_percent", "V_ppm"]], seed)
    print("split", seed, "- dropna", round(dropna_score, 3),
          " filled", round(filled_again, 3),
          " gap", round(dropna_score - filled_again, 3))

The first split was a flattering one: its gap is the third biggest of the ten,
and across all ten the dropna tree averages 0.780 and the filled tree
0.726 — a gap of 0.054, half of what the first split
showed, running from 0.001 on split 8 to
0.113 on split 4. So: filling the holes does cost a few
points, part of that cost is the difficulty deletion was hiding, and no single split can tell you
the size of anything. One held-out third of 514 samples wobbles by more than the effect
you are trying to measure.

From here every number comes out of `accuracy`, so every number is measured on the same
756 samples and the same held-out third of them, and they can be compared.

## Do the badly measured columns help, or hurt?

The **major oxides** are the measurements that account for nearly the whole weight of the rock —
its silicon, titanium, aluminium, iron, calcium, magnesium, manganese, potassium and sodium. That
is nine elements and ten columns, because this file reports iron twice, as FeO and as Fe2O3. They
are the first thing anybody measures, so if tectonic setting is written anywhere it should be
written there.

There is an eleventh oxide in the file, phosphorus, and it is worth a moment because of how it is
written: `P2O5(wt%)`, in a naming style nobody else in the table uses, and blank
14.6 percent of the time. That is what a compilation of hundreds of published
tables looks like from the inside. We hold to the ten below, so that "the majors" means one fixed
list for the rest of the week, and phosphorus goes in with everything else.

In [ ]:
major_oxides = ["SiO2_wt_percent", "TiO2_wt_percent", "Al2O3_wt_percent", "Fe2O3_wt_percent",
                "FeO_wt_percent", "CaO_wt_percent", "MgO_wt_percent", "MnO_wt_percent",
                "K2O_wt_percent", "Na2O_wt_percent"]

print(missing[major_oxides].round(3))

Eight of the ten are missing from about one sample in eight. The two iron columns are the
exception, missing from more than half: this file keeps FeO and Fe2O3 as separate columns rather
than adding them together, and 412 of the 756 samples
(54.5 percent) carry neither. So even the best-measured ten are not a
complete ten.

Three models on those ten columns, one call each.

In [ ]:
tree_score = accuracy(DecisionTreeClassifier(random_state=0), basalts[major_oxides])
forest_score = accuracy(RandomForestClassifier(n_estimators=200, random_state=0),
                        basalts[major_oxides])
svm_score = accuracy(SVC(), basalts[major_oxides])

print(f"one tree:                 {tree_score:.3f}")
print(f"a forest:                 {forest_score:.3f}")
print(f"a support vector machine: {svm_score:.3f}")

Three models, three lines, one interface. Every classifier in scikit-learn takes `fit` and `score`
in exactly the shape logistic regression did, which is why two of these arrived today as three
lines rather than as two new subjects.

The two new ones are worth a sentence each. A **random forest**. Ask a hundred slightly different
trees and take a vote. Each tree sees a different random sample of the rocks and a different random
handful of the columns, so each gets it wrong somewhere different, and the vote cancels the private
mistakes out — 0.921 against the single tree's 0.850 on the same
ten columns. And an **SVM**. Of all the lines separating the two groups, take the one leaving the
widest gap. That is why the scaler in `accuracy` matters — a gap is a distance, and a distance
measured across columns with wildly different units is meaningless.

The forest wins here, so the forest is what we take forward.

### Predict before you run

There are 51 chemistry columns in this file and you have just used ten of them. The
other 41 are trace elements, isotope ratios and the phosphorus you left behind,
and you counted them a moment ago: 19 of the 51 are more than half
empty, one of them (Sn_ppm) is missing 97.1 percent of the time,
and every hole in every one of them is about to be filled in with a median that no laboratory ever
measured.

Hand the forest all 51 columns instead of the ten. Better, or worse? Commit to a
number before you run anything — you will write it down in the next cell.

### ✏️ Your turn 4

Set `my_guess` to the accuracy you expect from all the columns. Then call `accuracy` once, on
`basalts[feature_columns]` with a fresh `RandomForestClassifier(n_estimators=200, random_state=0)`,
and print your guess, that score, and the difference between it and the `forest_score` the ten
oxides already got two cells above.

**Use these names**, because the self-check looks for them: `my_guess`, `all_score`.

In [ ]:
# ← your answer here


assert 0 <= my_guess <= 1, "my_guess is an accuracy, so a number between 0 and 1"
assert 0 < all_score < 1, \
    "all_score is an accuracy, a fraction of 1 - not a percentage"
assert abs(all_score - forest_score) > 0.01, \
    "that is the ten oxides' score again - this call takes basalts[feature_columns]"
print("✓ all", len(feature_columns), "columns —", round(all_score, 3), "against the ten oxides'",
      round(forest_score, 3), "- a difference of", round(all_score - forest_score, 3))

More columns won, by 0.044. Forty-one extra columns, most of them badly measured, several
of them nearly absent, all of their holes stuffed with medians — and the model got *better*.

The obvious worry is that one split of the samples flattered it. So do it again, four more times,
with a different random split each time.

In [ ]:
for seed in [0, 1, 2, 3, 4]:
    oxides_only = accuracy(RandomForestClassifier(n_estimators=200, random_state=0),
                           basalts[major_oxides], seed)
    everything = accuracy(RandomForestClassifier(n_estimators=200, random_state=0),
                          basalts[feature_columns], seed)
    print("split", seed, "- ten oxides", round(oxides_only, 3),
          " all columns", round(everything, 3),
          " gap", round(everything - oxides_only, 3))

All five splits point the same way, with gaps from 0.009 to
0.062. On split 1 the gap nearly closes, which is a useful
reminder of how much a single held-out third of 756 samples can wobble; but it never
reverses. The extra columns are carrying something.

## Is the forest reading the chemistry, or reading who measured the rock?

A fitted forest will tell you which columns its trees kept asking about. `feature_importances_` is
one number per column, and they add up to 1. Note that `accuracy` calls `fit` on the model object
you hand it, and `fit` changes that object, so after the call `forest` below is a trained forest and
can be interrogated.

In [ ]:
forest = RandomForestClassifier(n_estimators=200, random_state=0)
accuracy(forest, basalts[feature_columns])

importance = pd.Series(forest.feature_importances_, index=feature_columns)
importance = importance.sort_values(ascending=False)

for name in importance.head(10).index:
    print(f"{name:20s} importance {importance[name]:.3f}   missing {missing[name] * 100:.1f}%")

In [ ]:
top_ten = importance.head(10)
plt.barh(top_ten.index[::-1], top_ten.values[::-1])
plt.xlabel("share of the forest's decisions")
plt.ylabel("chemistry column")
plt.title("The 10 columns the forest leaned on, of 51 "
          "(fitted on 529 training basalts)")
plt.show()

Read that list against the petrology from the start of the notebook and it is not a random ten.
`Sr_ppm` and `Nb_ppm` are the two sides of the subduction signature: strontium
travels in the water coming off a sinking plate and niobium does not, so an arc basalt carries more
strontium than a ridge basalt and is conspicuously short of niobium, while an ocean island basalt,
which owes nothing to subduction, is the niobium-rich one. `TiO2_wt_percent` and
`Zr_ppm` are the titanium and zirconium that Pearce and Cann were already plotting by
hand in the 1970s, and titanium is half of Shervais's pair as well. The model has, on its own,
arrived at the elements a petrologist would have nominated.

It has also leaned hard on something a petrologist would want flagged. Strontium and potassium are
*mobile*: seawater and low-grade metamorphism move them around long after the rock has solidified,
so a mobile element can be telling you about the rock's later life rather than about the melt it
came from. Pearce and Cann used strontium anyway — titanium, zirconium and strontium was the second
of their two diagrams — and said in the same paper that alteration moves it, which is why titanium,
zirconium and yttrium is the diagram people trust on a rock that has sat under the ocean. The
forest cannot tell an altered rock from a fresh one. It found strontium the single most useful
column in the file, and on this compilation that works — but it is a reason to be careful about
handing this model an altered ocean-floor basalt.

And notice what is *not* in the top ten. The most useful of the 19 columns that are
more than half empty is `Fe2O3_wt_percent`, ranked 24; between them those
19 columns account for 0.097 of the forest's decisions. So the
forest's attention did not go to the emptiest columns. What it took from beyond the ten oxides were
trace elements like `Zr_ppm` and `Nb_ppm`, missing 16.5 and
35.6 percent of the time — patchy, but mostly there.

One more check before anybody reports 0.965 to a geochemist. *You got 99 percent. Be
suspicious. Did one of your columns already know the answer?*

Nothing in the chemistry knows the answer. But something else in this file might. Every sample came
from some published study, each study measured its own set of elements, and studies tend to be about
one setting at a time — a paper on Hawaiian volcanoes measures a particular list, a paper on the
Mariana arc a different one. If that is true then the *pattern of blanks* on a row is a fingerprint
of which paper the row came from, and the paper knows the setting.

`basalts[feature_columns].isna()` is a table of exactly the same shape, holding `True` where the
measurement is missing and `False` where it exists, and not one actual measurement. Hand the forest
that.

### ✏️ Your turn 5

First write down what you expect: set `my_blank_guess` to the accuracy you think a forest can reach
knowing only which numbers are missing.

Then build `blanks = basalts[feature_columns].isna()` and score a fresh forest on it with
`accuracy`. Do the same for `basalts[major_oxides].isna()`. Print both, and print your baseline from
your turn 1 for comparison.

Then, in the cell after, say in two or three sentences what those two numbers mean for the
`all_score` you reported in your turn 4, and which of your numbers you would hand a geochemist.

**Use these names**, because the self-check looks for them: `my_blank_guess`, `blanks`,
`blank_score`, `oxide_blank_score`.

In [ ]:
# ← your answer here


assert oxide_blank_score < blank_score, \
    "the ten oxides' blanks should say less than all the columns' blanks - same table twice?"
print("✓ the blanks alone —", round(blank_score, 3), "from all the columns and",
      round(oxide_blank_score, 3), "from the ten oxides, against a baseline of", round(baseline, 3))

*(Double-click this cell and replace this line with your answer.)*

## The question, answered

**From three different places, and the rock remembers which.** A ridge basalt melts by
decompression from mantle already stripped once; an arc basalt melts because water off a sinking
plate lowered the melting point, and arrives carrying the elements that travel in water and short
of the ones that do not; an ocean island basalt comes from hotter, less depleted mantle deeper down.
Those three histories leave three different chemistries, and a random forest reads them back with
0.921 accuracy from the ten major oxides alone, against 0.343 for
guessing. What it leans on hardest — strontium, titanium, zirconium, niobium — is exactly what the
melting mechanisms predict it should. The setting really is written in the chemistry; part of what
looks like extra skill from the other 41 columns is the file remembering who
measured what.

## Week 11 summary

**The question.** Where does a volcano get its magma?

### What to remember

| | |
|---|---|
| **1** | A rock's chemistry carries the tectonic setting it formed in. |
| **2** | Missing data is not useless data — a half-empty column can still carry real signal, and even the pattern of the blanks predicts the setting. |
| **3** | Deleting rows to avoid missing values can destroy a dataset entirely. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Feature importance** | How much of a forest's decisions each column carried — the model's own account of what it used. |
| **Missingness as data** | Which measurements are missing is itself a record of who measured the sample, and a model will use it if you let it. |
| **Decision tree** | A flowchart of yes/no questions, learned from the data instead of written by hand. |
| **Random forest** | Ask a hundred slightly different trees and take a vote. |
| **SVM** | Of all the lines separating the two groups, take the one leaving the widest gap. |
| **Imputation** | A blank is not a zero. Fill it with something defensible and say what you filled it with. |

### Code you met this week

| Function | What it does |
|---|---|
| `DecisionTreeClassifier(random_state=0)` | learn the flowchart of yes/no questions from the data |
| `RandomForestClassifier(n_estimators=200, random_state=0)` | a hundred slightly different trees, voting |
| `SVC()` | the widest-gap classifier; give it scaled columns or the gap means nothing |
| `SimpleImputer(strategy="median")` | put the middle value of a column into that column's holes |
| `StandardScaler()` | put every column on the same footing, so a distance across them means something |
| `filler.fit_transform(X_train) / filler.transform(X_test)` | learn the fill values on the training set only, then apply the same ones to the test set |
| `forest.feature_importances_` | one number per column: how much of the forest's decisions it carried |
| `plt.bar(x, heights) / plt.barh(labels, values)` | one bar per item; barh when the labels are words |

## Homework

Three parts, all on the same table and the same `accuracy` function from class.

Class showed you that the 51 columns together beat the ten major oxides. It never
asked what the badly measured columns can do on their own, never let you choose where to draw the
line, and never went back to `.dropna()` to see what it would have cost you.

Run the setup cell at the top of the notebook, then the cell below, and you have everything the
three parts need.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
missing = basalts[feature_columns].isna().sum() / len(basalts)
baseline = basalts["affinity"].value_counts().max() / len(basalts)
major_oxides = ["SiO2_wt_percent", "TiO2_wt_percent", "Al2O3_wt_percent", "Fe2O3_wt_percent",
                "FeO_wt_percent", "CaO_wt_percent", "MgO_wt_percent", "MnO_wt_percent",
                "K2O_wt_percent", "Na2O_wt_percent"]


def accuracy(model, features, seed=0):
    """Fit one model on one set of columns and score it on the held-out third of the samples."""
    labels = basalts["affinity"]
    # Split first, before anything at all is measured from the data. `stratify` keeps the mix
    # of affinities the same in both halves, so a rare one cannot land entirely in the test set.
    X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.3,
                                                        random_state=seed, stratify=labels)
    # Fill the holes, then put the columns on a common footing. Both of these learn their
    # numbers from the training half alone — `fit_transform` on it, plain `transform` on the
    # test half — because a median taken from rows you are about to be tested on is leakage.
    filler = SimpleImputer(strategy="median")
    X_train = filler.fit_transform(X_train)
    X_test = filler.transform(X_test)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    # Learn from the training half; report the score on the test half, which it has never seen.
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

### ✏️ Your turn 6

The 19 columns you counted in your turn 3 — the ones more than half empty — on
their own, with nothing else.

Build `sparse_columns` with a loop over `feature_columns`, keeping the names where
`missing[name] > 0.5`. Then score a `RandomForestClassifier(n_estimators=200, random_state=0)` on
`basalts[sparse_columns]` with `accuracy`. Print how many columns you kept, the score, and your
`baseline` from your turn 1.

Then answer it in one more printed line, quoting your score against your baseline: do
19 columns that are more than half empty carry real information about tectonic
setting, or not?

**Use these names**, because the self-check looks for them: `sparse_columns`, `sparse_score`.

In [ ]:
# ← your answer here


assert missing[sparse_columns].min() > 0.5, "every column in sparse_columns should be over half empty"
print("✓ the emptiest columns alone —", len(sparse_columns), "columns,",
      round(sparse_score, 3), "against a baseline of", round(baseline, 3))

### ✏️ Your turn 7

Your decision, and it is a real one. A geochemist might reasonably refuse to report a model that
leans on a column measured in three samples out of a hundred. So set a cutoff and throw away every
column emptier than it.

Set `cutoff` to **either 0.5 or 0.2** — your choice, and both are defensible. Build `kept_columns`
by looping over `feature_columns` and keeping the names where `missing[name] < cutoff`, and score a
forest on `basalts[kept_columns]`. Then do the same for the cutoff you did *not* pick, so that you
can say what your choice cost. Print, for each: the cutoff, how many columns survived and the
score. Class got 0.965 from all 51.

Then say it, in one more printed line: quote both scores and both column counts, and name what
your cutoff cost you.

**Use these names**, because the self-check looks for them: `cutoff`, `kept_columns`, `kept_score`.

In [ ]:
# ← your answer here


assert missing[kept_columns].max() < cutoff, "every column you kept should be emptier than cutoff"
print("✓ your cutoff —", cutoff, "keeps", len(kept_columns), "columns and scores",
      round(kept_score, 3))

### ✏️ Your turn 8

Back to `.dropna()`, the tool you started the day with. `basalts.dropna(subset=major_oxides)` drops
a row only when one of *those* columns is blank, and leaves holes elsewhere alone.

Print how many rows survive `basalts.dropna(subset=major_oxides)`. You already have the number to
set it against: your turn 3 found that dropping every row with a hole anywhere in it leaves 1
of the 756 samples.

Then, in the cell after, write two or three sentences using **those two counts**: what would you
have concluded about the 41 columns outside `major_oxides` — mostly trace elements and isotope
ratios — if `.dropna()` were the only tool you knew? Say what you would have measured, what you
would have reported, and which of this week's three takeaways that story would have broken.

**Use this name**, because the self-check looks for it: `rows_left_oxides`.

In [ ]:
# ← your answer here


assert rows_left_oxides > 1, \
    "ten columns should leave far more rows than all of them - did you pass subset=?"
print("✓ what dropna costs —", rows_left_oxides,
      "rows survive on the ten major oxides, against 1 on the whole table")

*(Double-click this cell and replace this line with your answer.)*